# Prepare merged_standardized dataset

This notebook validates, deduplicates, and creates fixed train/validation/test manifests for the larger `merged_standardized` dataset. It writes to `merged_../prepared_data` so the existing dataset workflow is unchanged.

## 1. Imports and configuration

In [5]:

from pathlib import Path
from collections import defaultdict
import hashlib
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image

from sklearn.model_selection import train_test_split

import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


PyTorch version: 2.5.1+cu121
CUDA available: True


In [6]:
PROJECT_CANDIDATES=[path for path in [Path.cwd(), *Path.cwd().parents] if (path/"brain_tumors").is_dir()]
if not PROJECT_CANDIDATES:
    raise FileNotFoundError("Could not locate the brain_tumors project folder.")
PROJECT_DIR=PROJECT_CANDIDATES[0]/"brain_tumors"
DATA_CANDIDATES=[PROJECT_DIR.parent/"merged_standardized",PROJECT_DIR/"merged_standardized",Path.cwd()/"merged_standardized"]
DATA_DIR=next((path for path in DATA_CANDIDATES if path.is_dir()),None)
if DATA_DIR is None:
    raise FileNotFoundError("Could not find the merged_standardized dataset folder.")
OUTPUT_DIR=PROJECT_DIR/"Larger_Dataset"/"merged_prepared_data"
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
CLASS_NAMES=["glioma_tumor","meningioma_tumor","normal","pituitary_tumor"]
VALID_EXTENSIONS={".jpg",".jpeg",".png",".bmp",".webp"}
print("Dataset directory:",DATA_DIR.resolve())
print("Output directory:",OUTPUT_DIR.resolve())


Dataset directory: C:\Users\rickv\School\Jaar3\minor\Challenge\Brain-Tumor\merged_standardized
Output directory: C:\Users\rickv\School\Jaar3\minor\Challenge\Brain-Tumor\brain_tumors\Larger_Dataset\merged_prepared_data


## 2. Discover and validate images

In [7]:
records=[]; invalid=[]
for class_name in CLASS_NAMES:
    class_dir=DATA_DIR/class_name
    if not class_dir.is_dir(): raise FileNotFoundError(f"Missing class folder: {class_dir.resolve()}")
    for path in sorted(class_dir.rglob("*")):
        if not path.is_file(): continue
        if path.suffix.lower() not in VALID_EXTENSIONS: continue
        try:
            with Image.open(path) as image:
                image.verify()
            with Image.open(path) as image:
                records.append({"filepath":str(path.resolve()),"filename":path.name,"class":class_name,"width":image.width,"height":image.height,"mode":image.mode,"format":image.format})
        except Exception as error:
            invalid.append({"filepath":str(path.resolve()),"error":str(error)})

df=pd.DataFrame(records)
if df.empty: raise ValueError("No valid images found.")
print("Readable images:",len(df))
print(df["class"].value_counts().reindex(CLASS_NAMES,fill_value=0))
print("Invalid images:",len(invalid))
if invalid: pd.DataFrame(invalid).to_csv(OUTPUT_DIR/"invalid_images.csv",index=False)

Readable images: 9715
class
glioma_tumor        2664
meningioma_tumor    2499
normal              1960
pituitary_tumor     2592
Name: count, dtype: int64
Invalid images: 0


## 3. Remove exact duplicates

In [8]:
def sha256(filepath):
    digest=hashlib.sha256()
    with open(filepath,"rb") as file:
        for chunk in iter(lambda:file.read(1024*1024),b""): digest.update(chunk)
    return digest.hexdigest()

df["sha256"]=df["filepath"].map(sha256)
conflicts=df.groupby("sha256")["class"].nunique()
conflicting_hashes=conflicts[conflicts>1].index.tolist()
if conflicting_hashes:
    raise ValueError(f"Identical files have conflicting labels: {len(conflicting_hashes)} hash groups")
unique_df=df.sort_values("filepath").drop_duplicates("sha256",keep="first").reset_index(drop=True)
print("Before deduplication:",len(df))
print("After deduplication:",len(unique_df))
unique_df.to_csv(OUTPUT_DIR/"unique_images.csv",index=False)

Before deduplication: 9715
After deduplication: 9715


## 4. Create stratified fixed splits

In [9]:
train_df,temp_df=train_test_split(unique_df,test_size=0.30,random_state=SEED,stratify=unique_df["class"])
val_df,test_df=train_test_split(temp_df,test_size=0.50,random_state=SEED,stratify=temp_df["class"])
train_df=train_df.copy(); val_df=val_df.copy(); test_df=test_df.copy()
train_df["split"]="train"; val_df["split"]="validation"; test_df["split"]="test"
splits_df=pd.concat([train_df,val_df,test_df],ignore_index=True)
print(f"Train: {len(train_df)} | Validation: {len(val_df)} | Test: {len(test_df)}")
print(pd.crosstab(splits_df["split"],splits_df["class"],margins=True))
splits_df.to_csv(OUTPUT_DIR/"data_splits.csv",index=False)
train_df.to_csv(OUTPUT_DIR/"train.csv",index=False)
val_df.to_csv(OUTPUT_DIR/"validation.csv",index=False)
test_df.to_csv(OUTPUT_DIR/"test.csv",index=False)
print("Saved manifests to:",OUTPUT_DIR.resolve())

Train: 6800 | Validation: 1457 | Test: 1458
class       glioma_tumor  meningioma_tumor  normal  pituitary_tumor   All
split                                                                    
test                 400               375     294              389  1458
train               1865              1749    1372             1814  6800
validation           399               375     294              389  1457
All                 2664              2499    1960             2592  9715
Saved manifests to: C:\Users\rickv\School\Jaar3\minor\Challenge\Brain-Tumor\brain_tumors\Larger_Dataset\merged_prepared_data


## 5. Leakage check

No SHA-256 hash may appear in more than one split.

In [10]:
split_hashes=splits_df.groupby("sha256")["split"].nunique()
leakage=split_hashes[split_hashes>1]
if len(leakage): raise ValueError(f"Data leakage detected in {len(leakage)} hash groups")
print("Leakage check passed.")

Leakage check passed.
